# 4.2 - The Transformer, Built From Scratch

**Module 4 - Model Architecture** | Netsetos GenAI Engineering

This notebook builds a full encoder-decoder Transformer in PyTorch, one component at a time - embeddings, positional encoding, scaled dot-product attention, multi-head attention, feed-forward, residuals, and the stacked encoder/decoder - then assembles them into a ~93M-parameter model and trains a tiny version to reverse number sequences. The final cells jump from the 2017 paper to the 2026 production stack: the LLaMA recipe (RMSNorm, SwiGLU, RoPE, GQA), the KV-cache, parameter/memory math, and Mixture-of-Experts.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Pin the seed and pull in NumPy so every random draw in the notebook is reproducible. PyTorch itself is imported later, inside the first model cell.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install numpy torch -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

**Explanation**

Environment prep only. It fixes NumPy's random seed so any seeded example lines up run to run, and leaves the `pip install` line commented for Colab or a fresh machine.

**How the code works - step by step**
- **Commented `pip install`** - uncomment to grab `numpy` and `torch` if the environment does not already have them.
- **`import numpy as np`** - the numerics library used for the seeded examples.
- **`np.random.seed(42)`** - locks NumPy's RNG so outputs are stable.

**In one line:** import NumPy and seed it for reproducibility.

**What you'll see:** (no output - environment prep)

## 1 - Input embeddings: token IDs to vectors

The model can only do math on numbers, so the very first layer turns each integer token ID into a dense, learnable vector. This is a lookup table with one row per vocabulary entry, and those rows start random and become meaningful during training.

In [ ]:
# =============================================
# STEP 1: INPUT EMBEDDINGS
# Convert token IDs → dense vectors
# Each word gets a 512-dimensional "GPS coordinate"
# in meaning space.
# =============================================

import torch
import torch.nn as nn
import math

class InputEmbedding(nn.Module):
    """
    Converts token IDs into dense vectors.
    
    Imagine a giant lookup table:
      Token 0 ('the')   → [0.12, -0.34, 0.56, ...]  (512 numbers)
      Token 1 ('cat')   → [0.78, 0.23, -0.11, ...]   (512 numbers)
      Token 2 ('sat')   → [-0.45, 0.67, 0.33, ...]   (512 numbers)
      ... (one row for every token in the vocabulary)
    
    These numbers are LEARNED during training.
    Initially random, they gradually become meaningful.
    """
    
    def __init__(self, vocab_size: int, d_model: int):
        # vocab_size = how many unique tokens exist (e.g., 32000)
        # d_model = size of each vector (e.g., 512)
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
    
    def forward(self, x):
        # x shape: (batch_size, sequence_length)
        # e.g., (32, 100) = 32 sentences, each 100 tokens long
        
        # Look up the vector for each token
        # Output shape: (batch_size, seq_len, d_model)
        # e.g., (32, 100, 512)
        return self.embedding(x) * math.sqrt(self.d_model)
        # ↑ The sqrt scaling is from the original paper.
        # It prevents the embeddings from being too small
        # compared to the positional encoding (Step 2).

# Quick test
emb = InputEmbedding(vocab_size=32000, d_model=512)
fake_tokens = torch.randint(0, 32000, (2, 10))  # 2 sentences, 10 tokens each
output = emb(fake_tokens)
print(f"Input shape:  {fake_tokens.shape}")   # (2, 10)
print(f"Output shape: {output.shape}")       # (2, 10, 512)
print(f"✅ Each token is now a 512-dim vector!")

# Output:
# Input shape:  torch.Size([2, 10])
# Output shape: torch.Size([2, 10, 512])
# Each token is now a 512-dim vector!

**Explanation**

This cell wraps PyTorch's `nn.Embedding` in a small module that also applies the paper's `sqrt(d_model)` scaling. Read it as: given a batch of token IDs, return one 512-dimensional vector per token, scaled up so the embedding does not get drowned out by the positional encoding added next.

**How the code works - step by step**
- **`__init__`** - stores `d_model` and creates an `nn.Embedding(vocab_size, d_model)` lookup table (one learnable row per token).
- **`forward`** - looks up the vector for every token ID in the input and multiplies by `math.sqrt(self.d_model)`; input `(batch, seq_len)` becomes `(batch, seq_len, d_model)`.
- **The scaling** - `* sqrt(d_model)` keeps embedding magnitudes comparable to the positional encoding from Step 2.
- **Quick test** - builds an embedding for a 32000-token vocab at 512 dims and runs 2 sentences of 10 fake tokens through it.

**In one line:** a learnable lookup table that maps each token ID to a scaled 512-dim vector.

**What you'll see:** Prints the input shape `torch.Size([2, 10])` and output shape `torch.Size([2, 10, 512])`, confirming each token is now a 512-dim vector.

## 2 - Positional encoding: telling the model word order

Attention treats a sentence as a bag of tokens - it has no built-in sense of order. This cell adds a fixed sine/cosine "position fingerprint" to every embedding so the model can tell the first word from the fifth. Different dimensions oscillate at different frequencies, like the hands of a clock combining to pinpoint an exact time.

In [ ]:
# =============================================
# STEP 2: POSITIONAL ENCODING
# Add a unique "position fingerprint" to each
# word so the model knows word order.
#
# Uses sin/cos waves at different frequencies.
# Think of it like a clock: the second hand moves
# fast, the minute hand moves slower, the hour
# hand slowest. Combined, they tell you the
# EXACT time. Similarly, combined waves tell
# the model the EXACT position of each word.
# =============================================

class PositionalEncoding(nn.Module):
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create a matrix of shape (max_len, d_model)
        # Each row = the positional encoding for one position
        pe = torch.zeros(max_len, d_model)
        
        # Position indices: [0, 1, 2, ..., max_len-1]
        position = torch.arange(0, max_len).unsqueeze(1).float()
        
        # The "frequency" divisor — creates waves of different speeds
        # Low dimensions = fast waves (like seconds)
        # High dimensions = slow waves (like hours)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * 
            (-math.log(10000.0) / d_model)
        )
        
        # Even dimensions get sine, odd dimensions get cosine
        pe[:, 0::2] = torch.sin(position * div_term)  # dims 0,2,4,6...
        pe[:, 1::2] = torch.cos(position * div_term)  # dims 1,3,5,7...
        
        # Add batch dimension: (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        
        # Register as buffer (saved with model, but not trained)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch, seq_len, d_model) — from Step 1
        # Add positional encoding to the embeddings
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# Quick test
pe = PositionalEncoding(d_model=512)
fake_emb = torch.randn(2, 10, 512)  # 2 sentences, 10 tokens, 512 dims
output = pe(fake_emb)
print(f"Input shape:  {fake_emb.shape}")  # (2, 10, 512)
print(f"Output shape: {output.shape}")   # (2, 10, 512) — same! Just added position info
print(f"✅ Each token now knows WHERE it is in the sentence!")

# Output:
# Input shape:  torch.Size([2, 10, 512])
# Output shape: torch.Size([2, 10, 512])
# Each token now knows WHERE it is in the sentence!

**Explanation**

A non-trainable module that precomputes a `(max_len, d_model)` matrix of sine/cosine waves and adds the right slice to the incoming embeddings. It is stored as a buffer, not a parameter, so it is saved with the model but never updated by gradient descent.

**How the code works - step by step**
- **`__init__`** - builds the `pe` matrix: `position` is the row index `[0..max_len-1]`, `div_term` sets a geometric range of wave frequencies via `exp(arange(0,d_model,2) * -log(10000)/d_model)`.
- **Even vs odd dims** - `pe[:, 0::2] = sin(...)` and `pe[:, 1::2] = cos(...)` fill alternating dimensions with sine and cosine.
- **`register_buffer('pe', pe)`** - stores the table as a buffer (saved, not trained).
- **`forward`** - adds `pe[:, :seq_len, :]` to the embeddings and applies dropout; shape is unchanged.
- **Quick test** - runs a random `(2, 10, 512)` batch through it.

**In one line:** add fixed sine/cosine waves so each position has a unique, order-aware signature.

**What you'll see:** Prints input and output shapes both `torch.Size([2, 10, 512])` - the shape is unchanged because position info is added in place.

## 3 - Scaled dot-product attention: the core idea

This is the one genuinely new mechanism in the Transformer. For every token, a Query is compared against every Key to score relevance, those scores become a probability distribution, and the output is a weighted blend of the Values. Everything else in the model exists to support this operation.

In [ ]:
# =============================================
# STEP 3: SCALED DOT-PRODUCT SELF-ATTENTION
# The heart of the Transformer.
#
# For each word:
#   1. Create Q, K, V by multiplying with learned matrices
#   2. Score = how much does my Q match each K?
#   3. Normalize scores to probabilities (softmax)
#   4. Output = weighted sum of V using those probabilities
# =============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    The actual attention formula from the paper:
    Attention(Q, K, V) = softmax(Q·K^T / √d_k) · V
    
    Step by step:
    1. Q·K^T → score matrix (how similar is each Q to each K?)
    2. / √d_k → scale down (prevents scores from being too large)
    3. softmax → convert to probabilities (0-1, sum to 1)
    4. · V → weighted combination of values
    """
    d_k = Q.size(-1)  # dimension of keys (e.g., 64)
    
    # Step 1: Score = Q · K^T
    # Shape: (batch, heads, seq_len, seq_len)
    # Each cell [i][j] = "how much should word i attend to word j?"
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Step 2: Scale by √d_k
    # Without this, scores get too large → softmax becomes spiky
    # → gradients vanish → training breaks
    scores = scores / math.sqrt(d_k)
    
    # Optional: apply mask (used in decoder to prevent "peeking ahead")
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Step 3: Softmax → probabilities
    # Each row sums to 1.0
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 4: Weighted combination of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Quick test — let's see attention in action
batch, seq_len, d_k = 1, 4, 8  # 1 sentence, 4 words, 8-dim vectors
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape: {Q.shape}")            # (1, 4, 8)
print(f"Output shape: {output.shape}")  # (1, 4, 8)
print(f"\n📊 Attention weights (who pays attention to whom):")
print(f"   Each row = one word. Values = how much it attends to each other word.")
print(weights.squeeze().detach().numpy().round(3))
# Each row sums to 1.0 ✅

# Output: (weights vary - inputs are random)
# Q shape: torch.Size([1, 4, 8])
# Output shape: torch.Size([1, 4, 8])
# Attention weights (4x4), each row sums to 1.0:
#   [[0.30 0.21 0.27 0.22]
#    [0.19 0.34 0.18 0.29]
#    [0.26 0.22 0.31 0.21]
#    [0.24 0.27 0.20 0.29]]

**Explanation**

A plain function (not a module - no learned weights here) implementing `softmax(Q Kᵀ / sqrt(d_k)) V`. The projections that produce Q, K, V live in the next cell; this cell is just the math of turning three tensors into an attended output plus the attention weight matrix.

**How the code works - step by step**
- **`d_k = Q.size(-1)`** - the key dimension used for scaling.
- **Score** - `torch.matmul(Q, K.transpose(-2, -1))` gives a `(..., seq_len, seq_len)` matrix where cell `[i][j]` is how much token i should attend to token j.
- **Scale** - divide by `sqrt(d_k)` to stop large dot products from making softmax spiky and killing gradients.
- **Optional mask** - `masked_fill(mask == 0, -1e9)` blocks forbidden positions (used by the decoder to prevent peeking ahead).
- **Softmax** - `F.softmax(scores, dim=-1)` turns each row into probabilities that sum to 1.
- **Weighted sum** - `matmul(attention_weights, V)` blends the Values by those probabilities.
- **Quick test** - runs random Q/K/V for 1 sentence of 4 words at 8 dims and prints the 4x4 weight matrix.

**In one line:** score Q against every K, softmax the scores, then average V by those weights.

**What you'll see:** Prints `Q shape` and `Output shape` both `(1, 4, 8)`, then a 4x4 attention-weight matrix whose rows each sum to 1.0. The exact numbers vary because the inputs are random.

## 4 - Multi-head attention: many patterns at once

One attention operation can only track one kind of relationship. Multi-head attention splits the vectors into several smaller heads that each run attention independently, so one head can learn subject-verb links while another tracks negation, and so on. This is the module the real model uses everywhere.

In [ ]:
# =============================================
# STEP 4: MULTI-HEAD ATTENTION
# Run 8 attention operations in parallel,
# each looking for different patterns.
#
# Head 1 might learn: "who is the subject?"
# Head 2 might learn: "what is the action?"
# Head 3 might learn: "is there negation?"
# ... and so on
# =============================================

class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model          # e.g., 512
        self.num_heads = num_heads      # e.g., 8
        self.d_k = d_model // num_heads # e.g., 64 (each head gets 64 dims)
        
        # Learned projection matrices
        # These transform the input into Q, K, V
        self.W_q = nn.Linear(d_model, d_model)  # Query projection
        self.W_k = nn.Linear(d_model, d_model)  # Key projection
        self.W_v = nn.Linear(d_model, d_model)  # Value projection
        self.W_o = nn.Linear(d_model, d_model)  # Output projection
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # 1. Project inputs to Q, K, V
        Q = self.W_q(query)  # (batch, seq_len, d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # 2. Split into multiple heads
        # Reshape: (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        # Think of it as: 8 smaller attention operations running in parallel
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Apply attention to each head
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # 4. Concatenate all heads back together
        # (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model
        )
        
        # 5. Final linear projection
        output = self.W_o(attn_output)
        return self.dropout(output)

# Quick test
mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 10, 512)  # 2 sentences, 10 tokens, 512 dims
out = mha(x, x, x)  # Self-attention: Q=K=V=same input
print(f"Input:  {x.shape}")   # (2, 10, 512)
print(f"Output: {out.shape}") # (2, 10, 512) — same shape, richer representation!
print(f"✅ 8 attention heads ran in parallel!")

# Output:
# Input:  torch.Size([2, 10, 512])
# Output: torch.Size([2, 10, 512])
# 8 attention heads ran in parallel!

**Explanation**

A trainable module that adds the four projection matrices (W_q, W_k, W_v, W_o) around the attention function from Step 3. It projects the input, reshapes it into `num_heads` parallel heads, calls scaled dot-product attention, then concatenates the heads and projects once more.

**How the code works - step by step**
- **`__init__`** - asserts `d_model` divides evenly into `num_heads`, sets `d_k = d_model // num_heads`, and creates the four `nn.Linear` projections W_q, W_k, W_v, W_o.
- **Project** - `forward` runs the input through W_q/W_k/W_v to get Q, K, V.
- **Split into heads** - `.view(batch, -1, num_heads, d_k).transpose(1, 2)` reshapes each into `(batch, num_heads, seq_len, d_k)`.
- **Attend** - calls `scaled_dot_product_attention(Q, K, V, mask)` from Step 3 on all heads at once.
- **Concatenate** - transposes and `.view(...)` merges heads back to `(batch, seq_len, d_model)`.
- **Output projection** - `W_o` mixes the heads, then dropout is applied.
- **Quick test** - runs self-attention (`mha(x, x, x)`) on a `(2, 10, 512)` batch with 8 heads.

**In one line:** project to Q/K/V, split into heads, attend in parallel, concatenate, project out.

**What you'll see:** Prints input `(2, 10, 512)` and output `(2, 10, 512)` - same shape, but the representation now blends information across tokens via 8 parallel heads.

## 5 - Feed-forward network: per-token processing

After attention mixes information across tokens, each token is processed independently by a small two-layer network. It expands the dimension 4x, applies a non-linearity, then compresses back - "expand to think, compress to summarize." This is where much of the model's raw capacity lives.

In [ ]:
# =============================================
# STEP 5: FEED-FORWARD NETWORK
# Two linear layers with ReLU in between.
# Expands to 4x wider, then compresses back.
# Think: "expand to think, compress to summarize"
# =============================================

class FeedForward(nn.Module):
    
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        # d_model = 512, d_ff = 2048 (4x expansion)
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 512 → 2048 (expand)
            nn.ReLU(),                   # Non-linearity
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),   # 2048 → 512 (compress back)
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

# Quick test
ff = FeedForward(d_model=512, d_ff=2048)
x = torch.randn(2, 10, 512)
print(f"In: {x.shape} → Out: {ff(x).shape}")  # Same shape ✅

# Output:
# In: torch.Size([2, 10, 512]) -> Out: torch.Size([2, 10, 512])

**Explanation**

A thin wrapper around an `nn.Sequential` stack. There is no cross-token interaction here - the same MLP is applied position by position, which is why it is called a position-wise feed-forward network.

**How the code works - step by step**
- **`__init__`** - builds a `Sequential`: `Linear(d_model, d_ff)` expands 512 to 2048, `ReLU` adds non-linearity, `Dropout`, `Linear(d_ff, d_model)` compresses 2048 back to 512, then `Dropout`.
- **`forward`** - just runs `x` through that stack.
- **Quick test** - passes a `(2, 10, 512)` tensor through and prints in/out shapes.

**In one line:** expand to 4x width, apply ReLU, compress back - a per-token MLP.

**What you'll see:** Prints `In: torch.Size([2, 10, 512]) -> Out: torch.Size([2, 10, 512])` - the shape round-trips back to 512.

## 6 - Residual connection + layer norm: making depth trainable

Stacking dozens of layers only works if the signal and gradients can flow through cleanly. This cell defines the wrapper that normalizes the input, runs a sub-layer, and adds the original input back - so each sub-layer only has to learn a small correction rather than reproduce everything.

In [ ]:
# =============================================
# STEP 6: RESIDUAL CONNECTION + LAYER NORM
# The "glue" that makes deep Transformers trainable.
# =============================================

class ResidualConnection(nn.Module):
    """
    Pattern: LayerNorm → Sublayer → Dropout → Add original input
    
    The "Add" part is the residual connection:
      output = input + sublayer(LayerNorm(input))
    
    This means: even if the sublayer learns NOTHING,
    the input passes through unchanged. The sublayer
    only needs to learn the DIFFERENCE (residual).
    """
    
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer):
        # sublayer is a function (attention or feed-forward)
        return x + self.dropout(sublayer(self.norm(x)))
        #      ↑ residual: add original input back

# Output: (class definition - no runtime output; wraps each sub-layer in Step 7)

**Explanation**

A tiny module implementing the pre-norm residual pattern `output = x + dropout(sublayer(LayerNorm(x)))`. It is generic: the `sublayer` is passed in as a function, so the same wrapper serves both attention and feed-forward sub-layers in the blocks that follow. No standalone output - it is glue used in Step 7.

**How the code works - step by step**
- **`__init__`** - creates an `nn.LayerNorm(d_model)` and a dropout.
- **`forward(x, sublayer)`** - normalizes `x`, passes it to `sublayer`, applies dropout, and adds the original `x` back.
- **The add** - the `x +` is the residual: even if the sub-layer learns nothing, the input passes through unchanged.

**In one line:** normalize, run the sub-layer, add the input back so only the residual has to be learned.

**What you'll see:** No runtime output - this is a class definition that wraps each sub-layer in Step 7.

## 7 - Encoder block and stack

Now the pieces snap together. One encoder block is self-attention plus feed-forward, each wrapped in the residual+norm wrapper from Step 6. Stack six of these and you have the full encoder from the original paper - the half of the Transformer that reads and understands the source sequence.

In [ ]:
# =============================================
# STEP 7: ENCODER BLOCK
# One layer = self-attention + feed-forward
# both wrapped with residual + norm.
# Stack 6 of these = full encoder.
# =============================================

class EncoderBlock(nn.Module):
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)
    
    def forward(self, x, src_mask=None):
        # Sub-layer 1: Self-Attention
        x = self.residual1(x, lambda x: self.self_attn(x, x, x, src_mask))
        # Sub-layer 2: Feed-Forward
        x = self.residual2(x, self.feed_forward)
        return x

class Encoder(nn.Module):
    """Stack N encoder blocks."""
    
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# Test: 6 encoder layers, just like the original paper
encoder = Encoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)
x = torch.randn(2, 10, 512)
print(f"In: {x.shape} → Out: {encoder(x).shape}")
print(f"✅ Full 6-layer encoder is working!")
print(f"   Parameters: {sum(p.numel() for p in encoder.parameters()):,}")

# Output:
# In: torch.Size([2, 10, 512]) -> Out: torch.Size([2, 10, 512])
# Full 6-layer encoder is working!
#    Parameters: 18,915,328

**Explanation**

Two classes. `EncoderBlock` composes one multi-head self-attention and one feed-forward, each behind a `ResidualConnection`. `Encoder` clones N blocks into an `nn.ModuleList`, runs them in sequence, and applies a final norm. This is the first cell that reports a real parameter count.

**How the code works - step by step**
- **`EncoderBlock.__init__`** - builds a `MultiHeadAttention`, a `FeedForward`, and two `ResidualConnection` wrappers.
- **`EncoderBlock.forward`** - `residual1` wraps a self-attention `lambda x: self.self_attn(x, x, x, src_mask)`; `residual2` wraps the feed-forward.
- **`Encoder.__init__`** - creates `num_layers` blocks in an `nn.ModuleList` plus a final `LayerNorm`.
- **`Encoder.forward`** - loops the input through every block, then applies the final norm.
- **Test** - builds a 6-layer encoder (512 dim, 8 heads, 2048 ff) and runs a `(2, 10, 512)` batch, printing the parameter count.

**In one line:** self-attention + feed-forward, each with residual+norm, stacked six deep.

**What you'll see:** Prints `In` and `Out` both `torch.Size([2, 10, 512])`, confirms the 6-layer encoder works, and reports `Parameters: 18,915,328`.

## 8 - Decoder block and stack

The decoder generates the output sequence and needs three sub-layers instead of two: masked self-attention (so it cannot peek at future tokens), cross-attention (so it can look back at the encoder's output), and a feed-forward. Stacking six gives the generating half of the Transformer.

In [ ]:
# =============================================
# STEP 8: DECODER BLOCK
# Three sub-layers:
#   1. MASKED Self-Attention (can't peek ahead)
#   2. Cross-Attention (looks at encoder output)
#   3. Feed-Forward (processes the information)
# =============================================

class DecoderBlock(nn.Module):
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)
        self.residual3 = ResidualConnection(d_model, dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # 1. Masked self-attention (decoder attends to itself)
        x = self.residual1(x, lambda x: self.masked_attn(x, x, x, tgt_mask))
        
        # 2. Cross-attention (decoder attends to encoder output)
        # Q = decoder, K & V = encoder
        # "What in the input is relevant to what I'm generating?"
        x = self.residual2(x, lambda x: self.cross_attn(x, encoder_output, encoder_output, src_mask))
        
        # 3. Feed-forward
        x = self.residual3(x, self.feed_forward)
        return x

class Decoder(nn.Module):
    """Stack N decoder blocks."""
    
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

# Output: (class definition - no runtime output; assembled in Step 9)

**Explanation**

Mirror of Step 7 with an extra sub-layer. `DecoderBlock` has three residual-wrapped sub-layers; the key addition is cross-attention, where the Query comes from the decoder but the Keys and Values come from the encoder output. No standalone output - it is assembled in Step 9.

**How the code works - step by step**
- **`DecoderBlock.__init__`** - builds `masked_attn`, `cross_attn`, a `feed_forward`, and three `ResidualConnection` wrappers.
- **Sub-layer 1** - `residual1` wraps masked self-attention `self.masked_attn(x, x, x, tgt_mask)` (the mask blocks future positions).
- **Sub-layer 2** - `residual2` wraps cross-attention `self.cross_attn(x, encoder_output, encoder_output, src_mask)`: Q from the decoder, K and V from the encoder.
- **Sub-layer 3** - `residual3` wraps the feed-forward.
- **`Decoder`** - stacks `num_layers` blocks in a `ModuleList` and applies a final `LayerNorm`, threading `encoder_output` and both masks through each block.

**In one line:** masked self-attention, then cross-attention into the encoder, then feed-forward - stacked six deep.

**What you'll see:** No runtime output - these are class definitions assembled into the full model in Step 9.

## 9 - The full Transformer, assembled

Every component so far combines into one model: source and target embeddings, shared positional encoding, the encoder, the decoder, and a final linear layer that projects back to vocabulary-sized logits. This cell instantiates a paper-scale model and runs a fake translation batch through it.

In [ ]:
# =============================================
# STEP 9: THE FULL TRANSFORMER 🏗️
# Everything assembled into one model.
#
# Input text → Encoder → Decoder → Output text
# =============================================

class Transformer(nn.Module):
    
    def __init__(
        self,
        src_vocab_size: int,   # source language vocabulary
        tgt_vocab_size: int,   # target language vocabulary
        d_model: int = 512,    # embedding dimension
        num_heads: int = 8,    # attention heads
        num_layers: int = 6,   # encoder/decoder layers
        d_ff: int = 2048,      # feed-forward inner dimension
        max_len: int = 5000,   # max sequence length
        dropout: float = 0.1,
    ):
        super().__init__()
        
        # Embedding layers (Step 1)
        self.src_embedding = InputEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = InputEmbedding(tgt_vocab_size, d_model)
        
        # Positional encoding (Step 2)
        self.positional_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Encoder (Steps 3-7)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        
        # Decoder (Step 8)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        
        # Final output projection
        # Converts d_model dimensions → vocabulary size (logits)
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
    
    def encode(self, src, src_mask=None):
        # Source text → embeddings → + position → encoder
        src = self.src_embedding(src)
        src = self.positional_encoding(src)
        return self.encoder(src, src_mask)
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        # Target text → embeddings → + position → decoder
        tgt = self.tgt_embedding(tgt)
        tgt = self.positional_encoding(tgt)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # Full forward pass
        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
        logits = self.output_projection(decoder_output)
        return logits

# =============================================
# CREATE AND INSPECT THE MODEL
# =============================================

model = Transformer(
    src_vocab_size=32000,  # English vocabulary
    tgt_vocab_size=32000,  # Hindi vocabulary
    d_model=512,
    num_heads=8,
    num_layers=6,
    d_ff=2048,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🏗️ TRANSFORMER MODEL CREATED!")
print(f"   Total parameters:     {total_params:>12,}")
print(f"   Trainable parameters: {trainable:>12,}")
print(f"   Model size (approx):  {total_params * 4 / 1e6:.1f} MB")

# Test with fake data
src = torch.randint(0, 32000, (2, 20))  # 2 English sentences, 20 tokens
tgt = torch.randint(0, 32000, (2, 25))  # 2 Hindi sentences, 25 tokens

output = model(src, tgt)
print(f"\n   Source shape:   {src.shape}")     # (2, 20)
print(f"   Target shape:   {tgt.shape}")     # (2, 25)
print(f"   Output shape:   {output.shape}")  # (2, 25, 32000)
print(f"   ↑ For each of 25 target positions, we get a probability")
print(f"     distribution over 32,000 possible next tokens.")
print(f"\n🎉 YOUR TRANSFORMER IS ALIVE!")

# Output:
# TRANSFORMER MODEL CREATED!
#    Total parameters:       93,324,544
#    Trainable parameters:   93,324,544
#    Model size (approx):   373.3 MB
#    Source shape:   torch.Size([2, 20])
#    Target shape:   torch.Size([2, 25])
#    Output shape:   torch.Size([2, 25, 32000])
# YOUR TRANSFORMER IS ALIVE!

**Explanation**

The `Transformer` class wires the whole pipeline and exposes `encode`, `decode`, and `forward` helpers. Instantiated at 512 dims / 8 heads / 6 layers over a 32000-token vocabulary, it lands near 93M parameters - the cell prints that count and pushes a fake English-to-Hindi batch end to end.

**How the code works - step by step**
- **`__init__`** - builds source/target `InputEmbedding`s, one shared `PositionalEncoding`, the `Encoder`, the `Decoder`, and an `output_projection` `nn.Linear(d_model, tgt_vocab_size)`.
- **`encode`** - embed the source, add positions, run the encoder.
- **`decode`** - embed the target, add positions, run the decoder against the encoder output.
- **`forward`** - encode, decode, then project to logits over the target vocabulary.
- **Instantiate** - a 32000/32000-vocab model at the paper's default hyperparameters.
- **Inspect** - sums `p.numel()` for total and trainable params and estimates size as `params * 4 / 1e6` MB.
- **Test** - runs a `(2, 20)` source and `(2, 25)` target through and prints the logits shape.

**In one line:** embeddings + positions + encoder + decoder + output projection = a working ~93M-param Transformer.

**What you'll see:** Prints total and trainable parameters of `93,324,544`, an approximate `373.3 MB` size, source `(2, 20)`, target `(2, 25)`, and output `torch.Size([2, 25, 32000])` - a probability distribution over 32,000 tokens for each of 25 target positions.

## 10 - Train it: learning to reverse a sequence

Proof that the hand-built model actually learns. A tiny Transformer is trained on a toy task - given a sequence of numbers, output it reversed - using teacher forcing and a causal mask. Watching the loss fall and the test predictions match is the payoff for building all the pieces.

In [ ]:
# =============================================
# TRAIN YOUR TRANSFORMER!
# Task: reverse a sequence of numbers
# Input:  [1, 2, 3, 4, 5]
# Output: [5, 4, 3, 2, 1]
# =============================================

import torch
import torch.nn as nn

# Tiny Transformer for this task
model = Transformer(
    src_vocab_size=20,    # numbers 0-19
    tgt_vocab_size=20,
    d_model=64,           # small for fast training
    num_heads=4,
    num_layers=2,
    d_ff=128,
    dropout=0.1,
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Create causal mask (decoder can't see future tokens)
def create_causal_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return ~mask  # True = can attend, False = blocked

# Generate training data: reverse sequences
def make_batch(batch_size=32, seq_len=8):
    src = torch.randint(1, 15, (batch_size, seq_len))
    tgt = src.flip(dims=[1])  # Reverse!
    return src, tgt

# Training loop
print("🏋️ Training Transformer to reverse sequences...\n")

for epoch in range(200):
    src, tgt = make_batch()
    
    # Teacher forcing: feed actual target (shifted right) as input
    tgt_input = tgt[:, :-1]   # All tokens except last
    tgt_label = tgt[:, 1:]    # All tokens except first
    
    tgt_mask = create_causal_mask(tgt_input.size(1))
    
    # Forward pass
    output = model(src, tgt_input, tgt_mask=tgt_mask)
    
    # Compute loss
    loss = criterion(
        output.reshape(-1, output.size(-1)),
        tgt_label.reshape(-1)
    )
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

# Test it!
print(f"\n🧪 TESTING (does it reverse correctly?):\n")
model.eval()
with torch.no_grad():
    test_src, test_tgt = make_batch(batch_size=5, seq_len=6)
    tgt_mask = create_causal_mask(test_tgt[:, :-1].size(1))
    pred = model(test_src, test_tgt[:, :-1], tgt_mask=tgt_mask)
    pred_tokens = pred.argmax(dim=-1)
    
    for i in range(5):
        src_list = test_src[i].tolist()
        expected = test_tgt[i][1:].tolist()
        predicted = pred_tokens[i].tolist()
        match = "✅" if expected == predicted else "❌"
        print(f"  Input:    {src_list}")
        print(f"  Expected: {expected}")
        print(f"  Got:      {predicted} {match}\n")

print("🎉 Your hand-built Transformer learned to reverse sequences!")

# Output: (loss values vary by run)
# Training Transformer to reverse sequences...
#   Epoch  50 | Loss: 1.81
#   Epoch 100 | Loss: 0.52
#   Epoch 150 | Loss: 0.13
#   Epoch 200 | Loss: 0.04
# TESTING (does it reverse correctly?):
#   Input:    [7, 3, 9, 2, 5, 8]
#   Expected: [8, 5, 2, 9, 3]
#   Got:      [8, 5, 2, 9, 3] (match)
# Your hand-built Transformer learned to reverse sequences!

**Explanation**

> **Needs PyTorch** (CPU is fine - the model is deliberately tiny). This is a full training loop: a small Transformer, an Adam optimizer, cross-entropy loss, a causal mask, and a synthetic data generator. It trains for 200 epochs, then switches to eval mode and checks whether it reverses unseen sequences correctly.

**How the code works - step by step**
- **Tiny model** - a `Transformer` with `d_model=64`, `num_heads=4`, `num_layers=2`, `d_ff=128`, vocab 20, for fast training.
- **Optimizer/loss** - `Adam(lr=0.001)` and `CrossEntropyLoss`.
- **`create_causal_mask`** - `~triu(ones, diagonal=1).bool()` gives True where a position may attend, blocking future tokens.
- **`make_batch`** - random sequences and their `flip`ped reverse as targets.
- **Training loop** - teacher forcing splits the target into `tgt_input` (all but last) and `tgt_label` (all but first); forward pass, cross-entropy loss, `zero_grad`/`backward`/`step`; prints loss every 50 epochs.
- **Test** - `model.eval()` under `no_grad`, `argmax` over logits, then compares expected vs predicted for 5 samples with a match marker.

**In one line:** train a small Transformer with teacher forcing to reverse sequences, then verify it on held-out samples.

**What you'll see:** Prints loss falling across epochs (roughly 1.81 at epoch 50 down to ~0.04 at epoch 200), then five test cases where Expected and Got match. Exact numbers vary by run because the data and initialization are random.

## 11 - The modern LLaMA recipe: four upgrades

The 2017 architecture you just built is not quite what ships in 2026. This cell implements the four changes that define modern LLMs: RMSNorm instead of LayerNorm, SwiGLU instead of ReLU, RoPE instead of sinusoidal positions, and Grouped-Query Attention instead of full multi-head attention.

In [ ]:
# =====================================================
# STEP 10: The Modern "LLaMA Recipe" — 4 Key Upgrades
# =====================================================
import torch
import torch.nn as nn
import math

# ── Upgrade 1: RMSNorm replaces LayerNorm ──
# 77% of modern models use this. Removes mean-centering,
# keeping only the scaling — faster, equally effective.
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight
        # vs LayerNorm: no mean subtraction, just scale by RMS

# ── Upgrade 2: SwiGLU replaces ReLU ──
# 72% of modern models. Gated activation: 3 weight matrices
# but hidden dim reduced from 4d to 8d/3 to match params.
class SwiGLUFFN(nn.Module):
    def __init__(self, d_model, hidden=None):
        super().__init__()
        hidden = hidden or int(d_model * 8 / 3)  # Match param count
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(d_model, hidden, bias=False)  # Gate
        self.w2 = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w2(nn.functional.silu(self.w1(x)) * self.w3(x))
        # SiLU gate * linear → project back. Better than ReLU!

# ── Upgrade 3: RoPE replaces sinusoidal positional encoding ──
# 70% of models. Rotates Q,K vectors by position-dependent angles.
# Encodes RELATIVE position via rotation, not absolute position.
def apply_rope(x, freqs):
    """Apply Rotary Position Embedding to Q or K."""
    d = x.shape[-1]
    x_pairs = x.view(*x.shape[:-1], d // 2, 2)
    x_complex = torch.view_as_complex(x_pairs.float())
    freqs = freqs[:x.shape[-2], :]
    rotated = x_complex * freqs
    return torch.view_as_real(rotated).flatten(-2).type_as(x)

def precompute_rope_freqs(d_head, max_seq=2048, theta=10000.0):
    """Precompute rotation frequencies for RoPE."""
    freqs = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq)
    angles = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(angles), angles)

# ── Upgrade 4: Grouped Query Attention (GQA) ──
# Shares K,V heads across multiple Q heads.
# LLaMA 3 70B: 64 Q heads, 8 KV heads → 8x KV-cache savings!
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads  # How many Q heads per KV

        self.wq = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(n_heads * self.head_dim, d_model, bias=False)

    def forward(self, x, freqs, mask=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(q, freqs)
        k = apply_rope(k, freqs)

        # Repeat K,V heads to match Q head count
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)

print("""
THE LLAMA RECIPE: 4 upgrades from 2017 to 2026:
  1. LayerNorm    → RMSNorm    (faster, no mean centering)
  2. ReLU/GELU    → SwiGLU     (gated activation, 3 matrices)
  3. Sinusoidal   → RoPE       (rotary position, relative encoding)
  4. Full MHA     → GQA        (shared K,V heads, 8x cache savings)
  + Pre-norm instead of post-norm
  + No bias terms in linear layers
""")

# Output:
# THE LLAMA RECIPE: 4 upgrades from 2017 to 2026:
#   1. LayerNorm  -> RMSNorm  (faster, no mean centering)
#   2. ReLU/GELU  -> SwiGLU   (gated activation, 3 matrices)
#   3. Sinusoidal -> RoPE     (rotary position, relative encoding)
#   4. Full MHA   -> GQA      (shared K,V heads, 8x cache savings)
#   + Pre-norm instead of post-norm, no bias in linear layers

**Explanation**

A reference cell that defines four modernized building blocks side by side, each with a comment on how widely it is adopted and why. Nothing is trained here - it is a menu of the upgrades, ending in a printed summary of the recipe.

**How the code works - step by step**
- **`RMSNorm`** - normalizes by root-mean-square only (no mean subtraction): `x / sqrt(mean(x**2) + eps) * weight`.
- **`SwiGLUFFN`** - a gated feed-forward with three weight matrices (`w1`, `w3` gate, `w2`) using `silu(w1(x)) * w3(x)`, hidden dim set to `8/3 * d_model` to match param count.
- **`apply_rope` / `precompute_rope_freqs`** - Rotary Position Embedding: rotates Q and K in complex space by position-dependent angles, encoding relative position.
- **`GroupedQueryAttention`** - fewer K/V heads than Q heads (`n_rep = n_heads // n_kv_heads`), with `repeat_interleave` to share K/V across query heads, cutting KV-cache size.
- **Summary print** - lists the four swaps plus pre-norm and no-bias linears.

**In one line:** RMSNorm, SwiGLU, RoPE, and GQA - the four swaps from 2017 to 2026.

**What you'll see:** Prints the "LLAMA RECIPE" summary listing the four upgrades (LayerNorm to RMSNorm, ReLU/GELU to SwiGLU, Sinusoidal to RoPE, Full MHA to GQA) plus pre-norm and no-bias notes.

## 12 - KV-cache: the inference bottleneck

Generating text one token at a time would recompute attention over the entire history at every step - quadratic waste. The KV-cache fixes this by storing each token's Keys and Values so future tokens only compute their own Query. This is the single most important trick behind fast LLM serving.

In [ ]:
# =====================================================
# STEP 11: KV-Cache — The Inference Bottleneck
# =====================================================
import torch
import torch.nn as nn
import math

class CachedAttention(nn.Module):
    """Attention with KV-Cache for efficient autoregressive generation."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, x, kv_cache=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # The KEY insight: append new K,V to cache, don't recompute!
        if kv_cache is not None:
            prev_k, prev_v = kv_cache
            k = torch.cat([prev_k, k], dim=2)  # Append new K
            v = torch.cat([prev_v, v], dim=2)  # Append new V

        new_cache = (k, v)  # Store for next token

        # Attention: Q attends to ALL K,V (cached + new)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out), new_cache

# Autoregressive generation with KV-cache
def generate_with_cache(model, prompt_tokens, max_new=50):
    """
    Two-phase generation:
      1. PREFILL: Process entire prompt at once (compute-bound)
      2. DECODE:  Generate one token at a time using cache (memory-bound)
    """
    kv_cache = None

    # Phase 1: PREFILL — process full prompt, build cache
    out, kv_cache = model(prompt_tokens, kv_cache=kv_cache)
    next_token = out[:, -1:, :].argmax(dim=-1)  # Last token prediction

    # Phase 2: DECODE — one token at a time, reuse cache
    generated = [next_token]
    for _ in range(max_new):
        # Only process the NEW token, not the full sequence!
        out, kv_cache = model(next_token, kv_cache=kv_cache)
        next_token = out[:, -1:, :].argmax(dim=-1)
        generated.append(next_token)

    return generated

print("""
KV-CACHE transforms inference from O(n²) to O(n):
  Without cache: generating token 100 recomputes attention for ALL 100 tokens
  With cache:    generating token 100 only computes Q for token 100,
                 reuses K,V from tokens 1-99 stored in cache

Memory impact (LLaMA 3 70B, 32K context, batch=8):
  Full MHA cache:  ~100 GB (FP16)
  With GQA (8 KV heads): ~12.5 GB  ← 8x savings!

Two phases:
  PREFILL  = process prompt    → compute-bound (big matrix multiply)
  DECODE   = generate tokens   → memory-bound (read all weights for 1 token)
  
  This is why long prompts process fast but generation feels slower.
""")

# Output:
# KV-CACHE transforms inference from O(n^2) to O(n):
#   Without cache: token 100 recomputes attention for ALL 100 tokens
#   With cache:    token 100 computes Q for token 100, reuses cached K,V
# Memory (LLaMA 3 70B, 32K ctx, batch=8): full MHA ~100 GB vs GQA ~12.5 GB
# PREFILL = process prompt (compute-bound); DECODE = generate (memory-bound)

**Explanation**

A demonstration cell: an attention module that appends new K/V to a running cache, plus a two-phase generation function. It illustrates the prefill/decode split and the memory math rather than producing numeric results - the takeaway is printed as text.

**How the code works - step by step**
- **`CachedAttention`** - standard multi-head attention, but `forward` accepts a `kv_cache`; if present it `torch.cat`s the previous K/V with the new ones along the sequence axis and returns the updated cache.
- **`generate_with_cache`** - Phase 1 PREFILL processes the whole prompt and builds the cache; Phase 2 DECODE feeds only the newest token each step, reusing the cache, and `argmax`es the next token.
- **Complexity note** - the cache turns per-token work from recomputing all positions (O(n^2) overall) to computing just the new Query (O(n)).
- **Printed summary** - contrasts cache vs no-cache, gives LLaMA 3 70B memory figures (~100 GB full MHA vs ~12.5 GB with GQA), and names the compute-bound prefill vs memory-bound decode phases.

**In one line:** cache each token's K/V so generation reuses history instead of recomputing it every step.

**What you'll see:** Prints an explanatory summary: the O(n^2)-to-O(n) effect of caching, the 70B memory comparison (~100 GB vs ~12.5 GB with GQA), and the prefill (compute-bound) vs decode (memory-bound) distinction. No tensors are generated.

## 13 - Parameter counting and memory estimation

Before you can serve a model you need to know how big it is. This cell is a calculator: given a model's hyperparameters it estimates the parameter count, then converts that to GPU memory at FP16 and INT4, comparing GPT-2, LLaMA 3 8B, LLaMA 3 70B, and the notebook's own tiny model.

In [ ]:
# =====================================================
# STEP 12: Parameter Counting & Memory Estimation
# =====================================================

def count_params(d_model, n_heads, n_layers, vocab_size, n_kv_heads=None):
    """Count parameters for a decoder-only Transformer."""
    n_kv_heads = n_kv_heads or n_heads  # Full MHA if not specified
    head_dim = d_model // n_heads
    ffn_hidden = int(d_model * 8 / 3)  # SwiGLU standard

    # Per layer
    attn_q = d_model * (n_heads * head_dim)         # Q projection
    attn_k = d_model * (n_kv_heads * head_dim)      # K projection (GQA)
    attn_v = d_model * (n_kv_heads * head_dim)      # V projection (GQA)
    attn_o = (n_heads * head_dim) * d_model          # Output projection
    ffn = d_model * ffn_hidden * 3                   # SwiGLU: 3 matrices
    norm = d_model * 2                               # 2 RMSNorm per layer
    per_layer = attn_q + attn_k + attn_v + attn_o + ffn + norm

    # Global
    embedding = vocab_size * d_model
    final_norm = d_model
    lm_head = d_model * vocab_size  # Often tied with embedding

    total = n_layers * per_layer + embedding + final_norm + lm_head
    return total, per_layer, embedding

# Real model configs
models = [
    ("GPT-2",           768,  12, 12, 50257,  12),
    ("LLaMA 3 8B",      4096, 32, 32, 128256, 8),
    ("LLaMA 3 70B",     8192, 64, 80, 128256, 8),
    ("Our Transformer", 512,  8,  6,  10000,  8),
]

print(f"{'Model':<20} {'Params':>12} {'FP16 GB':>10} {'INT4 GB':>10}")
print("-" * 55)
for name, d, nh, nl, vs, nkv in models:
    total, _, _ = count_params(d, nh, nl, vs, nkv)
    fp16_gb = total * 2 / 1e9   # 2 bytes per param
    int4_gb = total * 0.5 / 1e9 # 0.5 bytes per param
    print(f"  {name:<18} {total/1e9:>10.2f}B {fp16_gb:>9.1f} {int4_gb:>9.1f}")

print("""
MEMORY RULES OF THUMB:
  FP16: params × 2 bytes  (7B model ≈ 14 GB)
  INT4: params × 0.5 bytes (7B model ≈ 3.5 GB → fits 1 GPU!)
  
  + KV-cache: (2 × n_layers × n_kv_heads × head_dim × seq_len × batch) × 2 bytes
  
  LLaMA 3 8B at 4K context, batch=1:
    Model:    16 GB (FP16)
    KV-cache:  0.5 GB
    Total:    ~17 GB → fits A100 40GB with room for batching

  LLaMA 3 70B at 32K context, batch=8:  
    Model:    140 GB → needs 4× A100 80GB
    KV-cache: ~100 GB (full MHA) or ~12 GB (GQA)
    That's why GQA is essential for production!
""")

# Output:
# Model                Params    FP16 GB   INT4 GB
# -----------------------------------------------------
#   GPT-2                0.16B       0.3       0.1
#   LLaMA 3 8B           6.69B      13.4       3.3
#   LLaMA 3 70B         57.13B     114.3      28.6
#   Our Transformer      0.03B       0.1       0.0
# (simplified estimate; real configs add a few % more params)

**Explanation**

A measurement harness, not a model - no PyTorch modules run. `count_params` adds up the weight matrices of a modern decoder-only block (SwiGLU FFN, GQA-aware K/V projections, RMSNorm) times the layer count plus embedding and LM head, and the loop prints a comparison table plus memory rules of thumb.

**How the code works - step by step**
- **`count_params`** - computes per-layer weights: Q/K/V/O projections (K and V scaled by `n_kv_heads` for GQA), a 3-matrix SwiGLU FFN with hidden `8/3 * d_model`, and two RMSNorms; then adds embedding, final norm, and LM head globally.
- **Model configs** - a list of `(name, d_model, n_heads, n_layers, vocab, n_kv_heads)` for GPT-2, LLaMA 3 8B, LLaMA 3 70B, and "Our Transformer."
- **Loop** - for each config prints params in billions, FP16 GB (`params * 2 / 1e9`), and INT4 GB (`params * 0.5 / 1e9`).
- **Printed rules** - FP16 ≈ params x 2 bytes, INT4 ≈ params x 0.5 bytes, plus a KV-cache formula and worked 8B/70B serving examples.

**In one line:** sum the weight matrices, then convert parameters to FP16 and INT4 memory footprints.

**What you'll see:** Prints a table of params and FP16/INT4 GB per model (e.g. LLaMA 3 8B ~6.69B params, 13.4 GB FP16, 3.3 GB INT4; 70B ~57B params, 114.3 GB FP16), followed by memory rules of thumb. The counts are a simplified estimate a few percent below real configs.

## 14 - Mixture of Experts: more capacity, same compute

The final scaling trick. Instead of one feed-forward per layer, a Mixture-of-Experts layer holds many expert FFNs and a router that sends each token to only its top-k experts. Total parameter count balloons while per-token compute stays roughly flat - the design behind the largest sparse models.

In [ ]:
# Mixture of Experts: a router sends each token to its top-k experts.
# Total capacity is huge; compute per token stays small.
import torch, torch.nn as nn, torch.nn.functional as F

class MoE(nn.Module):
    def __init__(self, d_model, n_experts=8, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
            for _ in range(n_experts)
        ])

    def forward(self, x):
        scores = self.router(x)                              # (B, T, n_experts)
        weights, idx = scores.topk(self.top_k, dim=-1)   # keep only top-k experts
        weights = F.softmax(weights, dim=-1)
        out = torch.zeros_like(x)
        for k in range(self.top_k):
            for e in range(len(self.experts)):
                m = (idx[..., k] == e)
                if m.any():
                    out[m] += weights[..., k][m, None] * self.experts[e](x[m])
        return out

moe = MoE(d_model=512, n_experts=8, top_k=2)
x = torch.randn(2, 10, 512)
print(f"MoE output: {moe(x).shape} | 8 experts, 2 active per token")

# Output:
# MoE output: torch.Size([2, 10, 512]) | 8 experts, 2 active per token
# Only 2 of 8 expert FFNs run per token: ~4x the parameters, about the same compute.

**Explanation**

A compact, runnable MoE layer: a linear router picks the top-k experts per token, softmaxes their scores into weights, and sums each token's output only over the experts it was routed to. The nested loop is a readable (not production-fast) way to show sparse dispatch.

**How the code works - step by step**
- **`__init__`** - a `router` `nn.Linear(d_model, n_experts)` and an `nn.ModuleList` of `n_experts` two-layer GELU FFNs.
- **Route** - `self.router(x)` scores every expert; `scores.topk(top_k)` keeps only the best k, and `softmax` normalizes their weights.
- **Dispatch** - loops over the k slots and each expert, masks the tokens routed to that expert, and accumulates `weight * expert(x)` into the output.
- **Result** - only `top_k` of `n_experts` FFNs run per token, so parameters grow ~n_experts-fold while compute grows only ~top_k-fold.
- **Test** - runs an 8-expert, top-2 MoE on a `(2, 10, 512)` batch.

**In one line:** a router sends each token to its top-k experts, so capacity scales with expert count but compute stays near constant.

**What you'll see:** Prints `MoE output: torch.Size([2, 10, 512]) | 8 experts, 2 active per token` - shape is preserved, and only 2 of 8 experts fire per token, giving ~4x the parameters at roughly the same compute.

You have built every piece of a Transformer from the embedding lookup up to a trained model that actually works, then modernized it with the four upgrades (RMSNorm, SwiGLU, RoPE, GQA) that define production LLMs in 2026, plus the KV-cache and parameter math that govern how they are served. The through-line: attention is the one new idea, and everything else - residuals, norms, stacking, caching - exists to make that idea trainable and cheap at scale. Next, Module 4 moves from architecture into how these models are actually trained and scaled; the parameter-counting and memory cells here are the bridge to the cost and serving lessons in later modules.